# 11 - Implementation Time (real-time feasibility, measured)

**Question:** on the machine this study was actually run on, how long does one full CRI update --
`CRISupervisor.step()`, i.e. estimating $\hat S_k$ *and* computing all five indicators, the
combination, the dynamic filter, the classification, and the adaptation scaling -- actually take,
and how does that compare to the $\Delta t=T_s$ sampling interval used throughout the case study?

This fills in the paper's implementation-time sentence (Sec. III-D / Practical Considerations)
with real, measured numbers instead of placeholders.

Method: `CRISupervisor.step()` is timed in isolation (`time.perf_counter_ns()`, one call at a
time) while being fed the *real* $(u_k,y_k)$ sequence from an actual closed-loop run of each of
the eight scenarios (not synthetic random data) -- so the benchmark exercises the exact code path,
window sizes, and margin/adaptability/noise computations used everywhere else in this study, with
the first 100 samples of each run discarded as warmup (window fill + interpreter/cache warmup).

Outputs: `results/results_implementation_time.csv`.

In [1]:
import sys, os, time, platform
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd

from cri_microgrid.config import (PlantParams, ControllerParams, ConstraintParams, CRIParams,
                                   ProbeParams, ProfileParams, SimulationParams, DEFAULT_D_u, DEFAULT_D_y)
from cri_microgrid import scenarios, sim
from cri_microgrid.cri import CRISupervisor
from cri_microgrid.sensitivity import estimate_sensitivity

RESULTS_DIR = os.path.abspath("../results")
os.makedirs(RESULTS_DIR, exist_ok=True)

pp = PlantParams()
cp = ControllerParams()
cons = ConstraintParams()
crip = CRIParams()
probep = ProbeParams()
prof = ProfileParams()
SEED = 1

## Environment

`platform.processor()` returns a technical CPU-family string on Windows, not the marketing name --
the friendly CPU name below was looked up once via `(Get-CimInstance Win32_Processor).Name` in
PowerShell on this machine and is hardcoded here for the record; everything else is queried live.

In [2]:
CPU_MODEL = "AMD Ryzen 9 5900HX with Radeon Graphics"  # from (Get-CimInstance Win32_Processor).Name

env = dict(
    python_version=platform.python_version(),
    numpy_version=np.__version__,
    pandas_version=pd.__version__,
    os_platform=platform.platform(),
    cpu_technical=platform.processor(),
    cpu_model=CPU_MODEL,
    Ts_s=pp.Ts,
    Ts_ms=pp.Ts * 1000.0,
    N=crip.N, n_u=4, n_y=7,
)
for k, v in env.items():
    print(f"{k:16s} {v}")

python_version   3.10.9
numpy_version    2.2.6
pandas_version   2.3.3
os_platform      Windows-10-10.0.19045-SP0
cpu_technical    AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
cpu_model        AMD Ryzen 9 5900HX with Radeon Graphics
Ts_s             0.02
Ts_ms            20.0
N                40
n_u              4
n_y              7


## Isolated timing of one full CRI update, on the real per-scenario data stream

Each of the eight scenarios is re-run once (strategy `"cri"`, seed matching every other notebook
in this study), and every `CRISupervisor.step()` call is timed individually. This is deliberately
*not* a synthetic random-data microbenchmark: probing on/off transitions, real window content, and
the actual margin/adaptability/noise-variance computations for every scenario (including the
topology-outage window of S7 and the combined stress of S8) are all exercised exactly as they were
for every other result in this repository.

In [3]:
scenario_builders = {
    "S1": lambda: scenarios.build_S1(pp, prof, T_end=40.0),
    "S2": lambda: scenarios.build_S2(pp, prof, T_end=40.0),
    "S3": lambda: scenarios.build_S3(pp, prof, T_end=40.0),
    "S4": lambda: scenarios.build_S4(pp, prof, T_end=40.0),
    "S5": lambda: scenarios.build_S5(pp, prof, ramp_kind="fast", T_end=40.0),
    "S6": lambda: scenarios.build_S6(pp, prof, noise_level_pct=1.0, T_end=40.0),
    "S7": lambda: scenarios.build_S7(pp, prof, T_end=40.0, kind="vsc2_outage"),
    "S8": lambda: scenarios.build_S8(pp, prof, T_end=50.0),
}

N_WARMUP = 100

per_scenario_rows = []
all_times_ms = []

for sname, builder in scenario_builders.items():
    scenario = builder()
    simp = SimulationParams(T_end=scenario.T_end, seed=SEED, strategy="cri")
    log = sim.run_closed_loop(scenario, pp, cp, cons, crip, probep, simp)

    u_seq = np.stack([log["P1_cmd"], log["Q1_cmd"], log["P2_cmd"], log["Q2_cmd"]], axis=1)
    y_seq = np.stack([log["V1"], log["V2"], log["Vdc"], log["I1"], log["I2"], log["P_POI"], log["Q_POI"]], axis=1)
    n = len(log["t"])

    cri_sup = CRISupervisor(crip, D_u=DEFAULT_D_u, D_y=DEFAULT_D_y, n_u=4, n_y=7)
    times_ns = np.zeros(n - N_WARMUP)
    for k in range(n):
        t0 = time.perf_counter_ns()
        cri_sup.step(u_seq[k], y_seq[k], cons)
        t1 = time.perf_counter_ns()
        if k >= N_WARMUP:
            times_ns[k - N_WARMUP] = t1 - t0

    times_ms = times_ns / 1e6
    all_times_ms.append(times_ms)
    per_scenario_rows.append(dict(
        scenario=sname, n_samples=len(times_ms),
        mean_ms=times_ms.mean(), std_ms=times_ms.std(ddof=1),
        p95_ms=np.percentile(times_ms, 95), max_ms=times_ms.max(),
    ))

per_scenario = pd.DataFrame(per_scenario_rows).set_index("scenario")
display(per_scenario.round(4))

,n_samples,mean_ms,std_ms,p95_ms,max_ms
scenario,,,,,
S1,1901,0.2287,0.0162,0.2555,0.4368
S2,1901,0.2292,0.0238,0.2582,0.8592
S3,1901,0.2246,0.0192,0.2517,0.5326
S4,1901,0.2319,0.0197,0.2582,0.4798
S5,1901,0.2330,0.0229,0.2709,0.5410
S6,1901,0.2745,0.0790,0.4769,0.6515
S7,1901,0.2209,0.0224,0.2547,0.4676
S8,2401,0.2339,0.0165,0.2551,0.4544


## Result: pooled across all eight scenarios

In [4]:
allt = np.concatenate(all_times_ms)
Ts_ms = pp.Ts * 1000.0

summary = dict(
    n_total_samples=len(allt),
    mean_ms=float(allt.mean()),
    std_ms=float(allt.std(ddof=1)),
    median_ms=float(np.median(allt)),
    p95_ms=float(np.percentile(allt, 95)),
    p99_ms=float(np.percentile(allt, 99)),
    max_ms=float(allt.max()),
    Ts_ms=Ts_ms,
    mean_utilization_pct=float(100 * allt.mean() / Ts_ms),
    max_utilization_pct=float(100 * allt.max() / Ts_ms),
    p99_utilization_pct=float(100 * np.percentile(allt, 99) / Ts_ms),
)
for k, v in summary.items():
    print(f"{k:24s} {v:.4f}" if isinstance(v, float) else f"{k:24s} {v}")

print()
print(f"==> On {env['cpu_model']} (Python {env['python_version']}, NumPy {env['numpy_version']}):")
print(f"    one CRI update takes on average {summary['mean_ms']:.2f} ms "
      f"(maximum {summary['max_ms']:.2f} ms), "
      f"vs. a Delta t = {Ts_ms:.0f} ms sampling interval "
      f"-> {summary['mean_utilization_pct']:.1f}% mean / {summary['max_utilization_pct']:.1f}% worst-case occupancy.")

n_total_samples          15708
mean_ms                  0.2346
std_ms                   0.0367
median_ms                0.2274
p95_ms                   0.2742
p99_ms                   0.4544
max_ms                   0.8592
Ts_ms                    20.0000
mean_utilization_pct     1.1728
max_utilization_pct      4.2960
p99_utilization_pct      2.2719

==> On AMD Ryzen 9 5900HX with Radeon Graphics (Python 3.10.9, NumPy 2.2.6):
    one CRI update takes on average 0.23 ms (maximum 0.86 ms), vs. a Delta t = 20 ms sampling interval -> 1.2% mean / 4.3% worst-case occupancy.


## Cross-check: isolated sensitivity-estimate-only cost (the theoretical "dominant operation")

Sec. III-D of the paper argues that the regularized sensitivity estimate is the *asymptotically*
dominant operation ($\mathcal{O}(n_yn_uN+n_u^2N+n_u^3)$, Eq. 47), with everything else "only vector
operations and threshold comparisons." That is an accurate statement of asymptotic complexity, but
worth checking directly against the measured numbers above: at this study's actual (small) $N=40$,
$n_u=4$, $n_y=7$, does `estimate_sensitivity` alone actually account for most of the full-step
cost, or does per-call bookkeeping (deque-to-array stacking, margin/adaptability/noise-variance
calculations) make up a non-trivial share at this scale? Using `timing.benchmark_cri_step` (the
same isolated, synthetic-data benchmark used in `notebooks/10_statistics_and_complexity.ipynb`),
run at the exact case-study dimensions:

In [5]:
from cri_microgrid.timing import benchmark_cri_step

sens_only = benchmark_cri_step(N=crip.N, n_u=4, n_y=7, n_warmup=300, n_measure=3000, seed=1, lambda_s=crip.lambda_s)
sens_only_ms = {k: v / 1000.0 for k, v in sens_only["timing_us"].items()}
print("estimate_sensitivity alone (synthetic data, isolated):")
for k, v in sens_only_ms.items():
    print(f"  {k:6s} {v:.4f} ms")

share_pct = 100 * sens_only_ms["mean"] / summary["mean_ms"]
print(f"\nestimate_sensitivity is {share_pct:.0f}% of the full CRISupervisor.step() mean cost "
      f"at N={crip.N}, n_u=4, n_y=7 -- the remaining share is deque/window bookkeeping plus the "
      f"margin, adaptability, and noise-variance computations, all still O(N) or better per call.")

estimate_sensitivity alone (synthetic data, isolated):
  mean   0.0620 ms
  std    0.0085 ms
  p95    0.0668 ms
  max    0.2015 ms

estimate_sensitivity is 26% of the full CRISupervisor.step() mean cost at N=40, n_u=4, n_y=7 -- the remaining share is deque/window bookkeeping plus the margin, adaptability, and noise-variance computations, all still O(N) or better per call.


In [6]:
per_scenario.to_csv(os.path.join(RESULTS_DIR, "results_implementation_time_per_scenario.csv"))
pd.Series(summary).to_csv(os.path.join(RESULTS_DIR, "results_implementation_time_summary.csv"))
pd.Series(env).to_csv(os.path.join(RESULTS_DIR, "results_implementation_time_env.csv"))
print("Saved results/results_implementation_time_per_scenario.csv, _summary.csv, _env.csv")

Saved results/results_implementation_time_per_scenario.csv, _summary.csv, _env.csv


## Takeaway

Measured directly (not estimated) on the machine this entire study was run on, one full CRI
update -- the complete `CRISupervisor.step()`, not just the sensitivity estimate -- costs a mean
of well under half a millisecond and a worst-case (across 15,000+ samples spanning all eight
scenarios, including the topology-outage and combined-severe cases) of a few milliseconds, against
a $\Delta t=20$~ms supervisory sampling interval: comfortably real-time-feasible at this scale even
in an interpreted, general-purpose Python/NumPy implementation with no fixed-point or embedded-target
optimization applied. See `notebooks/11_bring_your_own.ipynb`'s (now `14_bring_your_own.ipynb`)
real-time porting checklist for what changes when moving this to an actual embedded target.